In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

## Coding for other models to perform testing

### AOD-Net

In [2]:
class AODNet(nn.Module):
    def __init__(self):
        super(AODNet, self).__init__()

        # Encoder Layers
        self.conv1 = nn.Conv2d(3, 3, kernel_size = 1)
        self.conv2 = nn.Conv2d(3, 3, kernel_size = 3, padding = 1)
        self.conv3 = nn.Conv2d(6, 3, kernel_size = 5, padding = 2)
        self.conv4 = nn.Conv2d(6, 3, kernel_size = 7, padding = 3)
        self.conv5 = nn.Conv2d(12, 3, kernel_size = 3, padding = 1)

        # ReLU activation
        self.relu = nn.ReLU(inplace = True)

        # Parameter b (bias term)
        self.b = 1.0

    def forward(self, x):
        # x is the input hazy image 
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(x1))

        # Concatenate features 
        cat1 = torch.cat((x1, x2), dim = 1)
        x3 = self.relu(self.conv3(cat1))

        cat2 = torch.cat((x2, x3), dim = 1)
        x4 = self.relu(self.conv4(cat2))

        cat3 = torch.cat((x1, x2, x3, x4), dim = 1)
        K = self.relu(self.conv5(cat3))

        # Apply the refrmulated atmospheric scattering model
        # J(x) = K(x) * I(x) - K(x) + b
        output = K * x - K + self.b

        return output

In [3]:
model = AODNet()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 1,761


### GridDehazeNet

In [4]:
class DownSample(nn.Module):
    def __init__(self, in_channels, channel_factor = 2, kernel_size = 3):
        super(DownSample, self).__init__()
        self.conv = nn.Conv2d(
            in_channels,
            in_channels * channel_factor, 
            kernel_size, 
            stride = 2,
            padding = kernel_size // 2   
        )

    def forward(self, x):
        return self.conv(x)

class UpSample(nn.Module):
    def __init__(self, in_channels, channel_factor = 2, kernel_size = 3):
        super(UpSample, self).__init__()
        self.conv = nn.Conv2d(
            in_channels, 
            in_channels // channel_factor,
            kernel_size, 
            padding = kernel_size // 2
        )
        
    def forward(self, x):
        x = F.interpolate(x, scale_factor = 2, mode = "bilinear", align_corners = True)
        return self.conv(x)
        

In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding = 1)
        self.relu = nn.ReLU(inplace = True)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding = 1)

    def forward(self, x):
        residual = x
        out = self.relu(self.conv1(x))
        out = self.conv2(out)
        out += residual 

        return self.relu(out)

CBAM Block

In [6]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction = 16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias = False),
            nn.ReLU(inplace = True),v
            nn.Conv2d(channels // reduction, channels, 1, bias = False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out

        return self.sigmoid(out) * x

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size = 7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding = kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim = 1, keepdim = True)
        max_out, _ = torch.max(x, dim = 1, keepdim = True)
        out = torch.cat([avg_out, max_out], dim = 1)
        out = self.conv(out)

        return self.sigmoid(out) * x

class CBAM(nn.Module):
    """Convolutional Block Attention Module"""
    def __init__(self, channels, reductions = 16):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(channels, reductions)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)

        return x

Grid Block

In [7]:
class GridBlock(nn.Module):
    def __init__(self, channels):
        super(GridBlock, self).__init__()
        self.res_block = ResidualBlock(channels)
        self.attention = CBAM(channels)

    def forward(self, x):
        x = self.res_block(x)
        x = self.attention(x)

        return x

In [8]:
class GridDehazeNet(nn.Module):
    def __init__(self, in_channels = 3, base_channels = 16, num_grid_blocks = 4):
        super(GridDehazeNet, self).__init__()

        # Preprocessing module
        self.pre_process = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        # Encoder (3 levels)
        self.down1 = DownSample(base_channels, channel_factor = 2)   # 32 channels
        self.down2 = DownSample(base_channels * 2, channel_factor = 2)  # 64 channels
        self.down3 = DownSample(base_channels * 4, channel_factor = 2)  # 128 channels

        # Grid blocks at each level
        # Level 0 (original resolution)
        self.grid_blocks_0 = nn.ModuleList([
            GridBlock(base_channels) for _ in range(num_grid_blocks)
        ])

        # Level 1 
        self.grid_blocks_1 = nn.ModuleList([
            GridBlock(base_channels * 2) for _ in range(num_grid_blocks)
        ])

        # Level 2
        self.grid_blocks_2 = nn.ModuleList([
            GridBlock(base_channels * 4) for _ in range(num_grid_blocks)
        ])

        # Level 3
        self.grid_blocks_3 = nn.ModuleList([
            GridBlock(base_channels * 8) for _ in range(num_grid_blocks)
        ])


        # Decoder 
        self.up3 = UpSample(base_channels * 8, channel_factor = 2)  # -> 64 channels
        self.up2 = UpSample(base_channels * 4, channel_factor = 2)  # -> 32 channels
        self.up1 = UpSample(base_channels * 2, channel_factor = 2)  # -> 16 channels

        # Post-processing
        self.post_process = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, in_channels, 3, padding=1)
        )

    def forward(self, x):
        x0 = self.pre_process(x)

        # Level 0 
        features_0 = x0
        for grid_block in self.grid_blocks_0:
            features_0 = grid_block(features_0)

        # Level 1
        x1 = self.down1(features_0)
        features_1 = x1
        for grid_block in self.grid_blocks_1:
            features_1 = grid_block(features_1)

        # Level 2
        x2 = self.down2(features_1)
        features_2 = x2
        for grid_block in self.grid_blocks_2:
            features_2 = grid_block(features_2)

        # Level 3
        x3 = self.down3(features_2)
        features_3 = x3
        for grid_block in self.grid_blocks_3:
            features_3 = grid_block(features_3)

        # Decoder path with skip connections
        up3 = self.up3(features_3)
        up3 = up3 + features_2

        up2 = self.up2(up3)
        up2 = up2 + features_1

        up1 = self.up1(up2)
        up1 = up1 + features_0

        # Post-process
        output = self.post_process(up1)

        # Residual connection with input
        output = output + x

        return output

In [9]:
# Create model
model = GridDehazeNet(in_channels=3, base_channels=16, num_grid_blocks=4)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 1,778,179
Trainable parameters: 1,778,179


### FFA-Net

Pixel and Channel Attention 

In [24]:
class PALayer(nn.Module):
    """Pixel Attention Layer"""
    def __init__(self, channels):
        super(PALayer, self).__init__()
        self.pa = nn.Sequential(
            nn.Conv2d(channels, channels // 8, 1, padding = 0, bias = True),
            nn.ReLU(inplace = True), 
            nn.Conv2d(channels // 8, 1, 1, padding = 0, bias = True),
            nn.Sigmoid()
        )

    def forward(self, x):
        y = self.pa(x)
        return x * y


class CALayer(nn.Module):
    """Channel Attention Layer"""
    def __init__(self, channels, reduction=8):
        super(CALayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, padding=0, bias=True),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y)
        return x * y

In [25]:
class BasicBlock(nn.Module):
    """Basic Block with Channel and Pixel Attention"""
    def __init__(self, channels, kernel_size = 3, reduction = 8):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size, padding = kernel_size // 2)
        self.relu = nn.ReLU(inplace = True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size, padding = kernel_size // 2)

        self.ca = CALayer(channels, reduction)
        self.pa = PALayer(channels)

    def forward(self, x):
        residual = x
        out = self.relu(self.conv1(x))
        out = self.conv2(out)
        out = self.ca(out)
        out = self.pa(out)
        out += residual 
        
        return out

In [35]:
class Group(nn.Module):
    def __init__(self, channels, kernel_size, blocks):
        super(Group, self).__init__()
        modules = [BasicBlock(channels, kernel_size) for _ in range(blocks)]
        modules.append(
            nn.Conv2d(channels, channels, kernel_size, padding = kernel_size // 2)
        )
        
        self.gp = nn.Sequential(*modules)
        
    def forward(self, x):
        res = self.gp(x)
        res += x
        return res
        

class FFA(nn.Module):
    def __init__(self, groups, blocks, channels = 64, kernel_size = 3):
        super(FFA, self).__init__()
        self.gps = groups
        self.channels = channels
        self.kernel_size = kernel_size

        self.preprocess = nn.Conv2d(3, channels, kernel_size, padding = kernel_size // 2)
        self.g1 = Group(channels, kernel_size, blocks = blocks)
        self.g2 = Group(channels, kernel_size, blocks = blocks)
        self.g3 = Group(channels, kernel_size, blocks = blocks)

        self.ca_layer = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(self.channels * self.gps, self.channels // 16, 1, padding=0),
            nn.ReLU(inplace = True),
            nn.Conv2d(self.channels // 16, self.channels * self.gps, 1, padding=0, bias=True),
            nn.Sigmoid()
        )

        self.pa_layer = PALayer(self.channels)

        self.postprocess = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding = kernel_size // 2),
            nn.Conv2d(channels, 3, kernel_size, padding = kernel_size // 2),
        )

    def forward(self, x):
        x1 = self.preprocess(x)
        g1 = self.g1(x1)
        g2 = self.g2(g1)
        g3 = self.g3(g2)

        w = self.ca_layer(torch.cat([g1, g2, g3], dim = 1))
        w = w.view(-1, self.gps, self.channels)[:, :, :, None, None]
        print(w.shape)
        out = w[:, 0, ...] * g1 + w[:, 1, ...] * g2 + w[:, 2, ...] * g3

        out = self.pa_layer(out)
        x1 = self.postprocess(out)

        return x1 + x

In [36]:
# Create model
model = FFA(groups=3, blocks=19)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

torch.Size([1, 3, 64, 1, 1])
Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 4,455,913
Trainable parameters: 4,455,913


## Testing all models

In [1]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
import sys 

sys.path.append("../")

In [19]:
from model.aecr_net import AECRNet
from model.aod_net import AODNet
from model.griddehaze_net import GridDehazeNet
from model.ffa_net import FFA
from model.msbdn import MSBDN
from model.dehamer import Dehamer 
from model.fsdgn import FSDGN
from model.fmphys_mamba import FM_PhysMamba_UNET, ODESolver 

#### AODNet Testing

In [3]:
# Create model
model = AODNet()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 1,761
Trainable parameters: 1,761


### GridDehazeNet Testing

In [4]:
# Create model
model = GridDehazeNet()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 1,778,179
Trainable parameters: 1,778,179


### AECRNet Testing

In [5]:
# Create model
model = AECRNet()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 713,910
Trainable parameters: 713,910


### FFA-Net Testing

In [6]:
# Create model
model = FFA()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

torch.Size([1, 3, 64, 1, 1])
Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 4,455,913
Trainable parameters: 4,455,913


### MSBDN Testing

In [7]:
# Create model
model = MSBDN()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 31,353,061
Trainable parameters: 31,353,061


### Dehamer Testing

In [10]:
# Create model
model = Dehamer()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

/workspace/dehazing/.venv/lib/python3.11/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 122,224,533
Trainable parameters: 122,224,533


### FSDGN Testing

In [12]:
# Create model
model = FSDGN()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example input
hazy_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image, _ = model(hazy_image)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 2,731,299
Trainable parameters: 2,731,299


### FMPhys_Mamba Testing

In [23]:
# Create model
model = FM_PhysMamba_UNET("../configs/model_cfgs/small.yaml")

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
ode_solver = ODESolver(model)

# Example input
# timestep = torch.rand(1, device=self.device)
hazy_image = torch.randn(1, 3, 256, 256).to(device)
# clean_image = torch.randn(1, 3, 256, 256).to(device)

# Forward pass
with torch.no_grad():
    dehazed_image = ode_solver.sample(hazy_image, nfe = 10)

print(f"Input shape: {hazy_image.shape}")
print(f"Output shape: {dehazed_image.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 37,266,695
Trainable parameters: 37,266,695


### DehazeDDPM 